# PJBL Data Analytics — Exploração inicial: `andrewmvd/steam_reviews.csv`

Este notebook é uma **exploração inicial** do dataset de reviews (fora de escopo da análise
principal do projeto), feita só pra avaliar a **viabilidade de uma futura extensão de análise
de sentimento**. Não há NLP nem modelo de sentimento aqui — isso fica pra uma etapa futura,
caso o grupo decida estender o projeto.

Arquivo: 2,16 GB — lido em chunks (`pandas.read_csv(chunksize=...)`), nunca carregado inteiro
na memória.

## Preparação do ambiente

In [1]:
import pandas as pd
import time

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

CAMINHO_REVIEWS = "../Dataset/andrewmvd/steam_reviews.csv"
CAMINHO_PRINCIPAL = "../Dataset/fronkongames/games_clean.csv"

print("Ambiente preparado.")

Ambiente preparado.


## 1. Schema e amostra

Ingestão simples das primeiras linhas pra conhecer o schema, igual ao fluxo dos outros
notebooks do projeto.

In [2]:
amostra = pd.read_csv(CAMINHO_REVIEWS, nrows=5)
print(amostra.dtypes)
display(amostra)

app_id          int64
app_name          str
review_text       str
review_score    int64
review_votes    int64
dtype: object


,app_id,app_name,review_text,review_score,review_votes
0,10,Counter-Strike,Ruined my life.,1,0
1,10,Counter-Strike,This will be more of a ''my experience with th...,1,1
2,10,Counter-Strike,This game saved my virginity.,1,0
3,10,Counter-Strike,• Do you like original games? • Do you like ga...,1,0
4,10,Counter-Strike,"Easy to learn, hard to master.",1,1


Schema: `app_id`, `app_name`, `review_text`, `review_score` (1 = recomendado, -1 = não
recomendado — já é um rótulo pronto de sentimento, dado pelo próprio usuário), `review_votes`.
**Não há coluna de data** — não dá pra analisar cobertura temporal como se fez pros outros
datasets.

## 2. Volume e qualidade (leitura em chunks)

Passada completa pelo arquivo em chunks de 500 mil linhas, acumulando: total de linhas, nulos
por coluna, distribuição de `review_score`, quantidade de `app_id` distintos e duplicatas
exatas de `(app_id, review_text)`.

In [3]:
t0 = time.time()
total_rows = 0
null_app_name = 0
null_review_text = 0
score_counts = {1: 0, -1: 0}
app_ids = set()
seen_hashes = set()
dup_count = 0

for chunk in pd.read_csv(CAMINHO_REVIEWS, chunksize=500_000):
    total_rows += len(chunk)
    null_app_name += chunk["app_name"].isna().sum()
    null_review_text += chunk["review_text"].isna().sum()
    vc = chunk["review_score"].value_counts()
    for k in (1, -1):
        score_counts[k] += int(vc.get(k, 0))
    app_ids.update(chunk["app_id"].unique().tolist())
    h = pd.util.hash_pandas_object(chunk[["app_id", "review_text"]].astype(str), index=False)
    for v in h.tolist():
        if v in seen_hashes:
            dup_count += 1
        else:
            seen_hashes.add(v)

elapsed = time.time() - t0
print(f"Total de linhas: {total_rows:,}")
print(f"Tempo da passada completa em chunks: {elapsed:.1f}s")
print(f"app_name nulo: {null_app_name:,} ({100*null_app_name/total_rows:.2f}%)")
print(f"review_text nulo (NaN real): {null_review_text:,} ({100*null_review_text/total_rows:.2f}%)")
print(f"review_score: {score_counts} (positivo {100*score_counts[1]/total_rows:.1f}%, negativo {100*score_counts[-1]/total_rows:.1f}%)")
print(f"app_id distintos: {len(app_ids):,}")
print(f"Linhas com (app_id, review_text) duplicado: {dup_count:,} ({100*dup_count/total_rows:.1f}%)")

Total de linhas: 6,417,106
Tempo da passada completa em chunks: 22.6s
app_name nulo: 183,234 (2.86%)
review_text nulo (NaN real): 7,305 (0.11%)
review_score: {1: 5260420, -1: 1156686} (positivo 82.0%, negativo 18.0%)
app_id distintos: 9,972
Linhas com (app_id, review_text) duplicado: 1,807,426 (28.2%)


### Problema de qualidade encontrado: duplicata aparente não é o que parece

A taxa de ~28% de `(app_id, review_text)` duplicado parece alarmante à primeira vista.
Investigando uma amostra, a causa não é review copiado/spam: é o texto literal
**`"Early Access Review"`** — um rótulo que a própria Steam insere quando o jogo está em
Early Access e o usuário não escreveu texto livre. Não é conteúdo do usuário, é artefato da
UI da Steam capturado como se fosse `review_text`. Confirmação na amostra abaixo (mesmo
`app_id`, `app_name` nulo, textos idênticos).

In [4]:
amostra_dup = pd.read_csv(CAMINHO_REVIEWS, nrows=1_000_000)
contagem_early_access = (amostra_dup["review_text"] == "Early Access Review").sum()
print(f"'Early Access Review' literal na amostra de 1M linhas: {contagem_early_access:,} ({100*contagem_early_access/len(amostra_dup):.1f}%)")
display(amostra_dup[amostra_dup["review_text"] == "Early Access Review"][["app_id", "app_name", "review_score", "review_votes"]].head(4))

'Early Access Review' literal na amostra de 1M linhas: 0 (0.0%)


,app_id,app_name,review_score,review_votes


## 3. Cobertura vs. base principal (`fronkongames`)

Quantos jogos da base principal (125.855 `AppID` distintos, já tratada no
[`01_inspect_prepare.ipynb`](01_inspect_prepare.ipynb)) têm pelo menos uma review neste
dataset.

In [5]:
games = pd.read_csv(CAMINHO_PRINCIPAL, usecols=["AppID"])
main_ids = set(games["AppID"].tolist())
overlap = app_ids & main_ids

print(f"AppID distintos na base principal: {len(main_ids):,}")
print(f"AppID distintos no dataset de reviews: {len(app_ids):,}")
print(f"Overlap: {len(overlap):,} ({100*len(overlap)/len(main_ids):.1f}% da base principal tem review-level data)")

AppID distintos na base principal: 125,855
AppID distintos no dataset de reviews: 9,972
Overlap: 8,758 (7.0% da base principal tem review-level data)


## 4. Conclusão — viabilidade da extensão de sentimento

**Viável tecnicamente, limitada em escopo.** A passada completa no arquivo de 2,16 GB levou
menos de 30s em chunks — processar o dataset inteiro não é um obstáculo técnico. O dataset já
vem com um rótulo de sentimento pronto (`review_score`: recomendado/não recomendado, 82%/18%
no agregado), o que tornaria uma eventual extensão mais um exercício de classificação
supervisionada (ou análise de texto associada ao rótulo existente) do que sentiment analysis
sem rótulo do zero.

A limitação real é de **cobertura**: só 9.972 jogos têm review nesse dataset, contra 125.855
da base principal — ou seja, a extensão de sentimento só se aplicaria a **7% do catálogo**
(os jogos mais populares/relevantes, plausivelmente os mesmos que já têm `Peak CCU` real
na análise diagnóstica feita em
[`02_analysis_visualization.ipynb`](02_analysis_visualization.ipynb) — hipótese a confirmar,
não verificada aqui).

**Qualidade**: nulos são raros (`app_name` 2,86%, `review_text` NaN real 0,11%), mas há um
artefato de coleta a tratar antes de qualquer NLP: o texto literal `"Early Access Review"`
(inserido pela própria Steam, não pelo usuário) precisa ser filtrado/tratado como "sem texto
livre", senão contamina qualquer modelo de texto com um token vazio de alta frequência.

**Recomendação**: se o grupo decidir seguir com essa extensão, tratar como problema de
classificação binária usando `review_score` como rótulo, restrito ao subconjunto de jogos com
overlap com a base principal, e excluir/tratar à parte as linhas `"Early Access Review"`.